In [1]:
import copy
import torch

In [2]:
class HebbNeuron:

    def __init__(self, input_size):
        # Pesos inicializados en cero
        self.w = torch.zeros(
            input_size,
            dtype=torch.float32
        )

        # Bias inicializado en cero
        self.b = torch.tensor(
            0.0,
            dtype=torch.float32
        )

    def forward(self, x):
        """
        Calcula la entrada neta:

        y_in = x1*w1 + x2*w2 + b

        Devuelve:
        1 si y_in >= 0
        -1 si y_in < 0
        """

        x = x.to(dtype=torch.float32)

        y_in = torch.sum(x * self.w) + self.b

        if y_in.item() >= 0:
            return 1

        return -1

    def train(self, S, T):
        """
        Entrenamiento mediante la regla de Hebb:

        w_nuevo = w_anterior + x*t
        b_nuevo = b_anterior + t
        """

        for i in range(len(S)):
            x = S[i].to(dtype=torch.float32)
            t = T[i].to(dtype=torch.float32)

            self.w = self.w + x * t
            self.b = self.b + t

In [3]:
class PerceptronNeuron:

    def __init__(self, input_size):
        # Se agrega una posición para el bias
        self.w = torch.zeros(
            input_size + 1,
            1,
            dtype=torch.float32
        )

        self.threshold = 0
        self.learning_rate = 0
        self.epochs = 0

    def forward(self, x):
        """
        w[0] = bias
        w[1] = peso de x1
        w[2] = peso de x2
        """

        x = x.to(dtype=torch.float32)

        # Agregar una entrada constante igual a 1 para el bias
        x_with_bias = torch.cat(
            (torch.tensor([1.0]), x)
        )

        # Entrada neta
        y_in = torch.matmul(
            x_with_bias,
            self.w
        ).item()

        # Función de activación con umbral simétrico
        if y_in > self.threshold:
            return 1

        if y_in < -self.threshold:
            return -1

        return 0

    def train(
        self,
        S,
        T,
        threshold=1,
        learning_rate=1,
        max_epochs=100
    ):
        """
        Entrena el perceptrón hasta completar una época
        sin errores o alcanzar max_epochs.
        """

        self.threshold = threshold
        self.learning_rate = learning_rate

        for epoch in range(max_epochs):

            hubo_error = False

            for i in range(len(S)):
                x = S[i].to(dtype=torch.float32)
                t = int(T[i].item())

                y = self.forward(x)

                # Actualizar sólo cuando la salida es incorrecta
                if y != t:
                    x_column = x.reshape(-1, 1)

                    self.w[1:] = (
                        self.w[1:]
                        + self.learning_rate
                        * x_column
                        * t
                    )

                    self.w[0] = (
                        self.w[0]
                        + self.learning_rate * t
                    )

                    hubo_error = True

            self.epochs = epoch + 1

            # Si no hubo errores, terminó el entrenamiento
            if not hubo_error:
                return True

        return False

In [4]:
# Entradas en representación bipolar
S = torch.tensor([
    [ 1,  1],     # Binario: 1, 1
    [ 1, -1],     # Binario: 1, 0
    [-1,  1],     # Binario: 0, 1
    [-1, -1]      # Binario: 0, 0
])

# Salidas esperadas para NOR
T_NOR = torch.tensor([
    -1,    # NOR(1,1) = 0
    -1,    # NOR(1,0) = 0
    -1,    # NOR(0,1) = 0
     1     # NOR(0,0) = 1
])

# Salidas esperadas para XNOR
T_XNOR = torch.tensor([
     1,    # XNOR(1,1) = 1
    -1,    # XNOR(1,0) = 0
    -1,    # XNOR(0,1) = 0
     1     # XNOR(0,0) = 1
])

print("Entradas:")
print(S)

print("\nTargets de NOR:")
print(T_NOR)

print("\nTargets de XNOR:")
print(T_XNOR)

Entradas:
tensor([[ 1,  1],
        [ 1, -1],
        [-1,  1],
        [-1, -1]])

Targets de NOR:
tensor([-1, -1, -1,  1])

Targets de XNOR:
tensor([ 1, -1, -1,  1])


In [5]:
hebb_nor = HebbNeuron(input_size=2)

hebb_nor.train(
    S=S,
    T=T_NOR
)

print("NOR con Hebb")
print("-------------------------")
print("w1 =", hebb_nor.w[0].item())
print("w2 =", hebb_nor.w[1].item())
print("Bias =", hebb_nor.b.item())

NOR con Hebb
-------------------------
w1 = -2.0
w2 = -2.0
Bias = -2.0


In [6]:
def bipolar_a_binario(valor):
    return 1 if int(valor) == 1 else 0


print("Tabla de verdad de NOR con Hebb")
print("----------------------------------------")
print("A\tB\tEsperado\tObtenido")

aciertos = 0

for x, target in zip(S, T_NOR):

    resultado = hebb_nor.forward(x)

    a = bipolar_a_binario(x[0])
    b = bipolar_a_binario(x[1])
    esperado = bipolar_a_binario(target)
    obtenido = bipolar_a_binario(resultado)

    if esperado == obtenido:
        aciertos += 1

    print(
        f"{a}\t{b}\t{esperado}\t\t{obtenido}"
    )

exactitud = aciertos / len(S) * 100

print("----------------------------------------")
print(f"Aciertos: {aciertos} de {len(S)}")
print(f"Exactitud: {exactitud:.2f}%")

Tabla de verdad de NOR con Hebb
----------------------------------------
A	B	Esperado	Obtenido
1	1	0		0
1	0	0		0
0	1	0		0
0	0	1		1
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [7]:
perceptron_nor = PerceptronNeuron(
    input_size=2
)

convergio = perceptron_nor.train(
    S=S,
    T=T_NOR,
    threshold=1,
    learning_rate=1,
    max_epochs=100
)

print("NOR con perceptrón")
print("-------------------------")
print("¿Convergió?", convergio)
print("Épocas =", perceptron_nor.epochs)
print("Bias =", perceptron_nor.w[0].item())
print("w1 =", perceptron_nor.w[1].item())
print("w2 =", perceptron_nor.w[2].item())
print("Umbral =", perceptron_nor.threshold)

NOR con perceptrón
-------------------------
¿Convergió? True
Épocas = 2
Bias = -2.0
w1 = -2.0
w2 = -2.0
Umbral = 1


In [8]:
print("Tabla de verdad de NOR con perceptrón")
print("----------------------------------------")
print("A\tB\tEsperado\tObtenido")

aciertos = 0

for x, target in zip(S, T_NOR):

    resultado = perceptron_nor.forward(x)

    a = bipolar_a_binario(x[0])
    b = bipolar_a_binario(x[1])
    esperado = bipolar_a_binario(target)
    obtenido = bipolar_a_binario(resultado)

    if esperado == obtenido:
        aciertos += 1

    print(
        f"{a}\t{b}\t{esperado}\t\t{obtenido}"
    )

exactitud = aciertos / len(S) * 100

print("----------------------------------------")
print(f"Aciertos: {aciertos} de {len(S)}")
print(f"Exactitud: {exactitud:.2f}%")

Tabla de verdad de NOR con perceptrón
----------------------------------------
A	B	Esperado	Obtenido
1	1	0		0
1	0	0		0
0	1	0		0
0	0	1		1
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [9]:
class XNORNetwork:

    def __init__(self, trained_nor_neuron):
        """
        La red utiliza cuatro copias de una neurona
        que ya aprendió la compuerta NOR.
        """

        self.nor_1 = copy.deepcopy(
            trained_nor_neuron
        )

        self.nor_2 = copy.deepcopy(
            trained_nor_neuron
        )

        self.nor_3 = copy.deepcopy(
            trained_nor_neuron
        )

        self.nor_4 = copy.deepcopy(
            trained_nor_neuron
        )

    def forward(self, x, show_steps=False):

        a = int(x[0].item())
        b = int(x[1].item())

        # Primera capa
        n1 = self.nor_1.forward(
            torch.tensor([a, b])
        )

        # Segunda capa
        n2 = self.nor_2.forward(
            torch.tensor([a, n1])
        )

        n3 = self.nor_3.forward(
            torch.tensor([n1, b])
        )

        # Capa de salida
        y = self.nor_4.forward(
            torch.tensor([n2, n3])
        )

        if show_steps:
            print(
                f"A={bipolar_a_binario(a)}, "
                f"B={bipolar_a_binario(b)} | "
                f"N1={bipolar_a_binario(n1)}, "
                f"N2={bipolar_a_binario(n2)}, "
                f"N3={bipolar_a_binario(n3)} | "
                f"XNOR={bipolar_a_binario(y)}"
            )

        return y

In [10]:
xnor_hebb = XNORNetwork(
    trained_nor_neuron=hebb_nor
)

print("XNOR construida con neuronas NOR de Hebb")
print("--------------------------------------------------")

for x in S:
    xnor_hebb.forward(
        x,
        show_steps=True
    )

XNOR construida con neuronas NOR de Hebb
--------------------------------------------------
A=1, B=1 | N1=0, N2=0, N3=0 | XNOR=1
A=1, B=0 | N1=0, N2=0, N3=1 | XNOR=0
A=0, B=1 | N1=0, N2=1, N3=0 | XNOR=0
A=0, B=0 | N1=1, N2=0, N3=0 | XNOR=1


In [11]:
print("Tabla de verdad de XNOR con Hebb")
print("----------------------------------------")
print("A\tB\tEsperado\tObtenido")

aciertos = 0

for x, target in zip(S, T_XNOR):

    resultado = xnor_hebb.forward(x)

    a = bipolar_a_binario(x[0])
    b = bipolar_a_binario(x[1])
    esperado = bipolar_a_binario(target)
    obtenido = bipolar_a_binario(resultado)

    if esperado == obtenido:
        aciertos += 1

    print(
        f"{a}\t{b}\t{esperado}\t\t{obtenido}"
    )

exactitud = aciertos / len(S) * 100

print("----------------------------------------")
print(f"Aciertos: {aciertos} de {len(S)}")
print(f"Exactitud: {exactitud:.2f}%")

Tabla de verdad de XNOR con Hebb
----------------------------------------
A	B	Esperado	Obtenido
1	1	1		1
1	0	0		0
0	1	0		0
0	0	1		1
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [12]:
xnor_perceptron = XNORNetwork(
    trained_nor_neuron=perceptron_nor
)

print("XNOR construida con perceptrones NOR")
print("--------------------------------------------------")

for x in S:
    xnor_perceptron.forward(
        x,
        show_steps=True
    )

XNOR construida con perceptrones NOR
--------------------------------------------------
A=1, B=1 | N1=0, N2=0, N3=0 | XNOR=1
A=1, B=0 | N1=0, N2=0, N3=1 | XNOR=0
A=0, B=1 | N1=0, N2=1, N3=0 | XNOR=0
A=0, B=0 | N1=1, N2=0, N3=0 | XNOR=1


In [13]:
print("Tabla de verdad de XNOR con perceptrón")
print("----------------------------------------")
print("A\tB\tEsperado\tObtenido")

aciertos = 0

for x, target in zip(S, T_XNOR):

    resultado = xnor_perceptron.forward(x)

    a = bipolar_a_binario(x[0])
    b = bipolar_a_binario(x[1])
    esperado = bipolar_a_binario(target)
    obtenido = bipolar_a_binario(resultado)

    if esperado == obtenido:
        aciertos += 1

    print(
        f"{a}\t{b}\t{esperado}\t\t{obtenido}"
    )

exactitud = aciertos / len(S) * 100

print("----------------------------------------")
print(f"Aciertos: {aciertos} de {len(S)}")
print(f"Exactitud: {exactitud:.2f}%")

Tabla de verdad de XNOR con perceptrón
----------------------------------------
A	B	Esperado	Obtenido
1	1	1		1
1	0	0		0
0	1	0		0
0	0	1		1
----------------------------------------
Aciertos: 4 de 4
Exactitud: 100.00%


In [14]:
print("=" * 60)
print("RESUMEN DEL EJERCICIO NOR Y XNOR")
print("=" * 60)

print("\nNOR con Hebb")
print("w1:", hebb_nor.w[0].item())
print("w2:", hebb_nor.w[1].item())
print("Bias:", hebb_nor.b.item())
print("Umbral implícito: 0")

print("\nNOR con perceptrón")
print("w1:", perceptron_nor.w[1].item())
print("w2:", perceptron_nor.w[2].item())
print("Bias:", perceptron_nor.w[0].item())
print("Umbral:", perceptron_nor.threshold)
print("Épocas:", perceptron_nor.epochs)

print("\nXNOR")
print("No se resolvió con una sola neurona.")
print("Se construyó con una red de cuatro neuronas NOR.")
print("Exactitud esperada: 100%")

RESUMEN DEL EJERCICIO NOR Y XNOR

NOR con Hebb
w1: -2.0
w2: -2.0
Bias: -2.0
Umbral implícito: 0

NOR con perceptrón
w1: -2.0
w2: -2.0
Bias: -2.0
Umbral: 1
Épocas: 2

XNOR
No se resolvió con una sola neurona.
Se construyó con una red de cuatro neuronas NOR.
Exactitud esperada: 100%
